# Integrated Gradients for Pairwise Ranking Explanations

This notebook implements Integrated Gradients (IG) to explain ranking decisions in a neural reranker.

Rather than explaining individual relevance scores, we explain **pairwise preferences** between two documents for a given query.

Given a query $q$ and two documents $d_i$ and $d_j$, the model assigns scores $s(q, d_i)$ and $s(q, d_j)$.

We define the pairwise preference function:

$$
g(q, d_i, d_j) = s(q, d_i) - s(q, d_j)
$$

If $g > 0$, the model prefers $d_i$ over $d_j$.

Integrated Gradients is applied to this function to understand which tokens contribute to *why one document is ranked above another*.

In [36]:
# -- IMPORTS --
import pickle
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm
import torch.nn.functional as F
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder
from captum.attr import IntegratedGradients

In [37]:
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
out_dir = Path("../outputs")
pairs_file = out_dir / "pairwise_scores.parquet"

# IG approximates the integral from baseline to input using N interpolated steps.
n_steps = 100

seed = 42
torch.manual_seed(seed)

## Model Setup

We use a transformer-based cross-encoder reranker that assigns a relevance score $s(q, d)$ to each query–document pair.

- Documents are scored independently
- Ranking is obtained by sorting scores
- Pairwise preferences are derived from score differences

This allows us to study pairwise explanations without requiring a model that explicitly takes document pairs as input.

In [38]:
ce_model = CrossEncoder(model_name, max_length=512)
bert_model = ce_model.model
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
bert_model = bert_model.to(device)
bert_model.eval()

print(f"Model device: {device}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

Model device: mps
Tokenizer vocab size: 30522


In [39]:
def load_pairs(path: Path):
    if path.suffix == ".parquet":
        try:
            return pd.read_parquet(path)
        except ImportError:
            fallback = path.with_suffix(".pkl")
            if fallback.exists():
                return pd.read_pickle(fallback)
            raise
    if path.suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported pair file format: {path}")

pairs_df = load_pairs(pairs_file)
print(f"Pairs loaded: {len(pairs_df)}")
print(f"Queries covered: {pairs_df['qid'].nunique()}")
pairs_df.head(3)

Pairs loaded: 90
Queries covered: 18


,qid,query,pid_i,passage_i,score_i,pid_j,passage_j,score_j,g_score,correct_pref
0,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,6875160,Home-schooling in Illinois is considered to be...,-8.177610,18.357065,1
1,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,4638437,Tuition and fees at Tulane University of Louis...,-3.351319,13.530774,1
2,1049774,cost of attendance eastern illinois university,7185662,"Eastern Illinois University has roughly 8,000 ...",10.179455,4453731,The average GPA at University of Illinois at C...,-6.636549,16.816004,1


## Tokenization

The cross-encoder expects input formatted as:

`[CLS] query [SEP] document [SEP]`

We tokenize query-document pairs accordingly, ensuring consistent input formatting for both scoring and attribution.

In [40]:
def tokenize_pair(query: str, passage: str, max_length: int = 512):
    """
    Tokenize a (query, passage) pair and keep track of the query/document split.
    """
    encoded = tokenizer(
        query,
        passage,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors="pt")

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    token_type_ids = encoded.get("token_type_ids")
    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(device)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
    sep_positions = (input_ids[0] == tokenizer.sep_token_id).nonzero(as_tuple=True)[0].tolist()
    sep_idx = sep_positions[0] if sep_positions else len(tokens) - 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
        "tokens": tokens,
        "sep_idx": sep_idx,
    }

## Forward Pass for Integrated Gradients

Captum's Integrated Gradients operates on embeddings rather than token IDs.

We define a forward function that:
- takes embeddings as input
- passes them through the model
- returns the relevance score

For pairwise explanations, we compute scores independently for each document and combine them as:

$$
g(q, d_i, d_j) = s(q, d_i) - s(q, d_j)
$$

In [41]:
embedding_layer = bert_model.get_input_embeddings()

def ids_to_embeds(input_ids):
    return embedding_layer(input_ids)

def forward_from_embeds(input_embeds, attention_mask, token_type_ids=None):
    """
    Runs the cross-encoder from precomputed embeddings and returns shape (batch,).
    """
    kwargs = {
        "inputs_embeds": input_embeds,
        "attention_mask": attention_mask,
    }

    if token_type_ids is not None:
        kwargs["token_type_ids"] = token_type_ids

    outputs = bert_model(**kwargs)
    return outputs.logits.squeeze(-1)

## Baseline

Integrated Gradients requires a baseline input representing "no information".

We construct the baseline by:
- replacing all content tokens with `[PAD]`
- keeping special tokens `[CLS]` and `[SEP]`

This provides a minimal-information reference input for attribution.

In [42]:
def make_baseline_input_ids(input_ids):
    """
    Replace content tokens with [PAD] while preserving structural special tokens.
    """
    if tokenizer.pad_token_id is None:
        raise ValueError("Tokenizer must define a pad token for the IG baseline.")

    baseline_ids = torch.full_like(input_ids, tokenizer.pad_token_id)
    special_token_ids = {
        token_id
        for token_id in [tokenizer.cls_token_id, tokenizer.sep_token_id]
        if token_id is not None
    }

    for pos, token_id in enumerate(input_ids[0].tolist()):
        if token_id in special_token_ids:
            baseline_ids[0, pos] = token_id

    return baseline_ids

def make_baseline_embeds(input_ids):
    baseline_ids = make_baseline_input_ids(input_ids)
    return ids_to_embeds(baseline_ids).detach()

## Aggregating Subword Attributions

BERT tokenizes words into subword pieces.

To improve interpretability:
- attribution scores are computed at token level
- subword scores are aggregated to word-level

This allows us to interpret importance in terms of human-readable words.

In [43]:
def merge_wordpieces(tokens, scores):
    special_tokens = set(tokenizer.all_special_tokens)
    word_tokens, word_scores = [], []
    current_word, current_score = "", 0.0

    for token, score in zip(tokens, scores):
        if token in special_tokens:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
                current_word, current_score = "", 0.0
            continue
        if token.startswith("##") and current_word:
            current_word += token[2:]
            current_score += score
        else:
            if current_word:
                word_tokens.append(current_word)
                word_scores.append(current_score)
            current_word = token.replace("##", "")
            current_score = score

    if current_word:
        word_tokens.append(current_word)
        word_scores.append(current_score)

    return word_tokens, np.array(word_scores)

def aggregate_attributions(attributions, tokens, sep_idx):
    """
    Convert embedding attributions into signed token- and word-level scores.
    Returns both full-sequence views and query/document splits.
    """
    token_scores = attributions[0].sum(dim=-1).detach().cpu().numpy()

    query_tokens = tokens[1:sep_idx]
    query_token_scores = token_scores[1:sep_idx]
    doc_tokens = tokens[sep_idx + 1:-1]
    doc_token_scores = token_scores[sep_idx + 1:-1]

    word_tokens, word_scores = merge_wordpieces(tokens, token_scores)
    query_word_tokens, query_word_scores = merge_wordpieces(query_tokens, query_token_scores)
    doc_word_tokens, doc_word_scores = merge_wordpieces(doc_tokens, doc_token_scores)

    return {
        "tokens": tokens,
        "token_scores": token_scores,
        "word_tokens": word_tokens,
        "word_scores": word_scores,
        "query_tokens": query_tokens,
        "query_token_scores": query_token_scores,
        "query_word_tokens": query_word_tokens,
        "query_word_scores": query_word_scores,
        "doc_tokens": doc_tokens,
        "doc_token_scores": doc_token_scores,
        "doc_word_tokens": doc_word_tokens,
        "doc_word_scores": doc_word_scores,
        "sep_idx": sep_idx,
    }

## Explanation Methods

We compare two explanation strategies:

### Method A: Pointwise Integrated Gradients (baseline)

We compute Integrated Gradients with respect to the pointwise score:

$$
s(q, d)
$$

for each document independently.

These explanations capture which tokens contribute to document relevance, but do not directly explain ranking decisions.

---

### Method B: Pairwise Integrated Gradients (proposed)

We compute Integrated Gradients with respect to the pairwise preference function:

$$
g(q, d_i, d_j) = s(q, d_i) - s(q, d_j)
$$

This aligns attribution with the ranking decision itself.

Note:
- The model scores documents independently
- Therefore, attribution signals decompose into contributions from each document
- The key difference lies in interpretation: explaining *preference* rather than *relevance*

In [44]:
def compute_pointwise_ig(query, passage):
    tok = tokenize_pair(query, passage)
    input_embeds = ids_to_embeds(tok["input_ids"]).detach()
    baseline_embeds = make_baseline_embeds(tok["input_ids"])

    ig = IntegratedGradients(forward_from_embeds)
    attributions, delta = ig.attribute(
        inputs=input_embeds,
        baselines=baseline_embeds,
        additional_forward_args=(tok["attention_mask"], tok["token_type_ids"]),
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    pointwise_results = {
        "method": "pointwise_ig",
        **aggregate_attributions(attributions, tok["tokens"], tok["sep_idx"]),
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
    }

    return pointwise_results

In [45]:
def paired_pointwise_ig(query, passage_i, passage_j, ig_i=None, ig_j=None):
    """
    Baseline: compute pointwise IG separately for (q, d_i) and (q, d_j),
    then compare them.
    
    For now, we return both sides separately rather than forcing token-level
    subtraction across different sequences.
    """
    if ig_i is None:
        ig_i = compute_pointwise_ig(query, passage_i)
    if ig_j is None:
        ig_j = compute_pointwise_ig(query, passage_j)

    pointwise_pair_baseline = {
        "method": "pointwise_pair_baseline",
        "doc_i": ig_i,
        "doc_j": ig_j}

    return pointwise_pair_baseline

### Method B: Pairwise Integrated Gradients (Implementation)

We define the target function:

$$
g(q, d_i, d_j) = s(q, d_i) - s(q, d_j)
$$

To compute this:

1. The model is run independently on:
   - $(q, d_i)$
   - $(q, d_j)$

2. Scores are combined into $g$

3. Integrated Gradients is applied to this function

The resulting attributions provide:

- **positive contributions** → support the preferred document
- **negative contributions** → support the competing document

This enables explanation of *why one document is ranked above another*.

In [46]:
def compute_pairwise_ig(query, passage_i, passage_j):
    tok_i = tokenize_pair(query, passage_i)
    tok_j = tokenize_pair(query, passage_j)

    input_embeds_i = ids_to_embeds(tok_i["input_ids"]).detach()
    input_embeds_j = ids_to_embeds(tok_j["input_ids"]).detach()
    baseline_embeds_i = make_baseline_embeds(tok_i["input_ids"])
    baseline_embeds_j = make_baseline_embeds(tok_j["input_ids"])

    def forward_pairwise(input_embeds_i, input_embeds_j,
                         attention_mask_i, token_type_ids_i,
                         attention_mask_j, token_type_ids_j):
        score_i = forward_from_embeds(input_embeds_i, attention_mask_i, token_type_ids_i)
        score_j = forward_from_embeds(input_embeds_j, attention_mask_j, token_type_ids_j)
        return score_i - score_j

    ig = IntegratedGradients(forward_pairwise)
    attributions, delta = ig.attribute(
        inputs=(input_embeds_i, input_embeds_j),
        baselines=(baseline_embeds_i, baseline_embeds_j),
        additional_forward_args=(
            tok_i["attention_mask"],
            tok_i["token_type_ids"],
            tok_j["attention_mask"],
            tok_j["token_type_ids"],
        ),
        n_steps=n_steps,
        return_convergence_delta=True,
    )

    attr_i, attr_j = attributions

    pairwise_results = {
        "method": "pairwise_ig",
        "doc_i": aggregate_attributions(attr_i, tok_i["tokens"], tok_i["sep_idx"]),
        "doc_j": aggregate_attributions(attr_j, tok_j["tokens"], tok_j["sep_idx"]),
        "convergence_delta": float(delta.detach().cpu().item()) if torch.is_tensor(delta) else float(delta),
    }

    return pairwise_results

## Test on a Single Pair

Before running the full experiment, we test both methods on a single example.

This verifies:
- correct computation of $g(q, d_i, d_j)$
- correct behavior of Integrated Gradients
- reasonable attribution outputs

In [47]:
# take the first pair with correct preference (g > 0) as the test case
test_row = pairs_df[pairs_df["correct_pref"] == 1].iloc[0]

print(f"Query: {test_row['query']}")
print(f"Passage i: {test_row['passage_i'][:120]}...")
print(f"Passage j: {test_row['passage_j'][:120]}...")
print(f"g score: {test_row['g_score']:.3f} (model prefers di)")

Query: cost of attendance eastern illinois university
Passage i: Eastern Illinois University has roughly 8,000 students. Admission is selective. Tuition is approximately $8,550 per year...
Passage j: Home-schooling in Illinois is considered to be a form of private education. Parents who choose to educate their children...
g score: 18.357 (model prefers di)


In [48]:
# run pointwise IG on di
test_pointwise_i = compute_pointwise_ig(test_row["query"], test_row["passage_i"])
test_pointwise_j = compute_pointwise_ig(test_row["query"], test_row["passage_j"])

print(f"Convergence delta di: {test_pointwise_i['convergence_delta']:.4f}")
print(f"Convergence delta dj: {test_pointwise_j['convergence_delta']:.4f}")

# show top-10 for di (pointwise)
idx = np.argsort(test_pointwise_i["word_scores"])[::-1][:10]
print("\nTop-10 words by pointwise IG attribution (di only):")
for i in idx:
    print(f"{test_pointwise_i['word_tokens'][i]:<20} {test_pointwise_i['word_scores'][i]:.4f}")

Convergence delta di: 0.0000
Convergence delta dj: 0.0000

Top-10 words by pointwise IG attribution (di only):
illinois             0.9209
eastern              0.5964
university           0.5640
illinois             0.4743
attendance           0.3823
eastern              0.3721
attendance           0.3043
admission            0.2156
residents            0.2123
university           0.1795


In [49]:
# run pairwise IG
test_pairwise = compute_pairwise_ig(
    test_row["query"],
    test_row["passage_i"],
    test_row["passage_j"])

print(f"Convergence delta: {test_pairwise['convergence_delta']:.4f}")

for side, label in [("doc_i", "passage_i"), ("doc_j", "passage_j")]:
    attr = test_pairwise[side]
    idx = np.argsort(attr["doc_word_scores"])[::-1][:10]
    print(f"\nTop-10 document words by pairwise IG attribution for {label}:")
    for i in idx:
        print(f"{attr['doc_word_tokens'][i]:<20} {attr['doc_word_scores'][i]:.4f}")

Convergence delta: 0.0000

Top-10 document words by pairwise IG attribution for passage_i:
illinois             0.9209
eastern              0.5964
university           0.5640
illinois             0.4743
attendance           0.3043
admission            0.2156
residents            0.2123
university           0.1795
has                  0.1655
$                    0.1483

Top-10 document words by pairwise IG attribution for passage_j:
schooling            1.1215
home                 0.6955
in                   0.2395
parents              0.2374
code                 0.2288
the                  0.2256
children             0.2254
.                    0.2161
section              0.1843
legal                0.1796


## Running IG on All Pairs

We compute explanations for all query–document pairs using:

- Pointwise IG (baseline)
- Pairwise IG (proposed method)

Results are stored for further analysis and evaluation.

In [50]:
attribution_records = []
failed_pairs = []

for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Computing IG"):
    try:
        # 1) pointwise IG (baseline)
        pt_ig_i = compute_pointwise_ig(row["query"], row["passage_i"])
        pt_ig_j = compute_pointwise_ig(row["query"], row["passage_j"])

        # 2) pointwise difference IG (baseline)
        pt_diff_ig = paired_pointwise_ig(
            row["query"],
            row["passage_i"],
            row["passage_j"],
            ig_i=pt_ig_i,
            ig_j=pt_ig_j)

        # 3) pairwise IG (proposed)
        pw_ig = compute_pairwise_ig(
            row["query"],
            row["passage_i"],
            row["passage_j"])

        attribution_records.append({
            "qid": row["qid"],
            "query": row["query"],
            "pid_i": row["pid_i"],
            "pid_j": row["pid_j"],
            "g_score": row["g_score"],
            "correct_pref": row["correct_pref"],
            # pointwise IG
            "pointwise_ig_i": pt_ig_i,
            "pointwise_ig_j": pt_ig_j,
            # pointwise difference IG
            "pointwise_difference_ig": pt_diff_ig,
            # pairwise IG
            "pairwise_ig": pw_ig})

    except Exception as e:
        failed_pairs.append({"qid": row["qid"], "pid_i": row["pid_i"], "error": str(e)})
        print(f"Error on qid={row['qid']}, pid_i={row['pid_i']}: {e}")

print(f"\nSuccessfully attributed: {len(attribution_records)} pairs")
if failed_pairs:
    print(f"Failed: {len(failed_pairs)} pairs")

Computing IG:   0%|          | 0/90 [00:00<?, ?it/s]


Successfully attributed: 90 pairs


In [51]:
pw_deltas = [r["pairwise_ig"]["convergence_delta"] for r in attribution_records]
pt_deltas = [r["pointwise_ig_i"]["convergence_delta"] for r in attribution_records]

print("Convergence delta- Pointwise IG (di):")
print(f" mean={np.mean(pt_deltas):.4f}, max={np.max(np.abs(pt_deltas)):.4f}")

print("Convergence delta- Pairwise IG:")
print(f" mean={np.mean(pw_deltas):.4f}, max={np.max(np.abs(pw_deltas)):.4f}")

if np.max(np.abs(pw_deltas)) > 0.05:
    print("\nWarning: some deltas are large. Consider increasing n_steps.")
else:
    print("\nAll deltas look good, approximation is accurate.")

Convergence delta- Pointwise IG (di):
 mean=0.0000, max=0.0000
Convergence delta- Pairwise IG:
 mean=0.0000, max=0.0001

All deltas look good, approximation is accurate.


## Qualitative Inspection

We inspect attribution outputs to ensure they are interpretable.

For each example:
- top positive words → support relevance or preference
- top negative words → reduce relevance or explain ranking loss

This helps validate that explanations align with intuition before quantitative evaluation.

In [52]:
def show_top_words(record, method="pairwise_ig", side=None, segment="doc", n=10):
    attr = record[method] if side is None else record[method][side]
    words = attr[f"{segment}_word_tokens"]
    scores = attr[f"{segment}_word_scores"]

    if len(words) == 0:
        print(f"No {segment} words available for {method}.")
        return

    top_pos = np.argsort(scores)[::-1][:n]
    top_neg = np.argsort(scores)[:n]
    side_label = f" ({side})" if side is not None else ""

    print(f"Method: {method}{side_label}")
    print(f"Query: {record['query']}")
    print(f"g_score: {record['g_score']:.3f}")

    print(f"\nTop-{n} positive {segment} words:")
    for i in top_pos:
        print(f" {words[i]:<25} {scores[i]:.4f}")

    print(f"\nTop-{n} negative {segment} words:")
    for i in top_neg:
        print(f" {words[i]:<25} {scores[i]:.4f}")

# show the first record
r = attribution_records[0]
print("=" * 60)
show_top_words(r, method="pointwise_ig_i", segment="doc")
print()
show_top_words(r, method="pairwise_ig", side="doc_i", segment="doc")
print()
show_top_words(r, method="pairwise_ig", side="doc_j", segment="doc")

Method: pointwise_ig_i
Query: cost of attendance eastern illinois university
g_score: 18.357

Top-10 positive doc words:
 illinois                  0.9209
 eastern                   0.5964
 university                0.5640
 illinois                  0.4743
 attendance                0.3043
 admission                 0.2156
 residents                 0.2123
 university                0.1795
 has                       0.1655
 $                         0.1483

Top-10 negative doc words:
 .                         -0.0567
 .                         -0.0550
 .                         -0.0536
 .                         -0.0426
 of                        -0.0406
 -                         -0.0249
 24                        -0.0231
 .                         -0.0223
 and                       -0.0197
 ,                         -0.0153

Method: pairwise_ig (doc_i)
Query: cost of attendance eastern illinois university
g_score: 18.357

Top-10 positive doc words:
 illinois                  0.9209


In [53]:
# find the pair with the smallest g_score (tightest decision)
closest_pair = pairs_df[pairs_df["correct_pref"] == 1].nsmallest(1, "g_score").iloc[0]
print(f"Tightest pair g_score: {closest_pair['g_score']:.3f}")
print(f"Query: {closest_pair['query']}")

# find it in attribution_records
r_tight = next(r for r in attribution_records 
               if r["pid_i"] == closest_pair["pid_i"] 
               and r["pid_j"] == closest_pair["pid_j"])

print("=" * 60)
show_top_words(r_tight, method="pointwise_ig_i", segment="doc")
print()
show_top_words(r_tight, method="pairwise_ig", side="doc_i", segment="doc")
print()
show_top_words(r_tight, method="pairwise_ig", side="doc_j", segment="doc")

Tightest pair g_score: 0.512
Query: what is qualfon
Method: pointwise_ig_i
Query: what is qualfon
g_score: 0.512

Top-10 positive doc words:
 qualfon                   3.1306
 is                        1.4706
 a                         0.7983
 global                    0.5126
 provider                  0.4037
 of                        0.2094
 globe                     0.2079
 outsourcing               0.2021
 intelligent               0.1835
 outsourcing               0.1768

Top-10 negative doc words:
 .                         -0.1357
 locations                 -0.0450
 services                  -0.0435
 in                        -0.0341
 )                         -0.0238
 .                         -0.0114
 and                       -0.0024
 we                        0.0019
 .                         0.0028
 process                   0.0030

Method: pairwise_ig (doc_i)
Query: what is qualfon
g_score: 0.512

Top-10 positive doc words:
 qualfon                   3.1306
 is            

In [54]:
# for each pair, compute the ratio of pairwise to pointwise magnitude
# this isolates the margin effect from document-specific effects

for label, r in [("Easy (g=18.357)", attribution_records[0]), ("Tight (g=0.512)", r_tight)]:
    pw_mag = np.abs(r["pairwise_ig"]["doc_i"]["doc_word_scores"]).mean()
    pt_mag = np.abs(r["pointwise_ig_i"]["doc_word_scores"]).mean()
    print(f"{label}: pairwise/pointwise magnitude ratio = {pw_mag/pt_mag:.3f}")

Easy (g=18.357): pairwise/pointwise magnitude ratio = 1.000
Tight (g=0.512): pairwise/pointwise magnitude ratio = 1.000


## Save Outputs

In [55]:
out_path = out_dir / "attributions.pkl"
with open(out_path, "wb") as f:
    pickle.dump(attribution_records, f)

print(f"Saved {len(attribution_records)} attribution records → {out_path}")

# save a summary csv
summary = pd.DataFrame([{
    "qid": r["qid"],
    "pid_i": r["pid_i"],
    "pid_j": r["pid_j"],
    "g_score": r["g_score"],
    "correct_pref": r["correct_pref"],
    "pw_ig_delta": r["pairwise_ig"]["convergence_delta"],
    "pt_ig_delta": r["pointwise_ig_i"]["convergence_delta"],
    "pw_top_doc_i_word": r["pairwise_ig"]["doc_i"]["doc_word_tokens"][np.argmax(r["pairwise_ig"]["doc_i"]["doc_word_scores"])] if len(r["pairwise_ig"]["doc_i"]["doc_word_tokens"]) > 0 else "",
    "pw_top_doc_j_word": r["pairwise_ig"]["doc_j"]["doc_word_tokens"][np.argmax(r["pairwise_ig"]["doc_j"]["doc_word_scores"])] if len(r["pairwise_ig"]["doc_j"]["doc_word_tokens"]) > 0 else "",
} for r in attribution_records])

summary.to_csv(out_dir / "attributions_summary.csv", index=False)
print(f"Saved summary CSV- {out_dir / 'attributions_summary.csv'}")
summary.head()

Saved 90 attribution records → ../outputs/attributions.pkl
Saved summary CSV- ../outputs/attributions_summary.csv


,qid,pid_i,pid_j,g_score,correct_pref,pw_ig_delta,pt_ig_delta,pw_top_doc_i_word,pw_top_doc_j_word
0,1049774,7185662,6875160,18.357065,1,0.000000,4.768372e-07,illinois,schooling
1,1049774,7185662,4638437,13.530774,1,0.000106,4.768372e-07,illinois,tulane
2,1049774,7185662,4453731,16.816004,1,0.000006,4.768372e-07,illinois,gpa
3,1049774,7185662,4760145,10.647701,1,0.000003,4.768372e-07,illinois,fisk
4,1049774,7185662,5305640,13.525833,1,0.000000,4.768372e-07,illinois,arkansas
